# RAID v5: Sentence Contrast Memory Curves

## Overview
This notebook measures whether distant context helps an LM distinguish the **real** next sentence from an impostor sentence drawn from a different document in the same genre.

## Method
For each document, we identify sentence boundaries in the target region (second half). At each boundary, we:
1. Provide W tokens of preceding context (W = 4, 8, 12, 16, 24, 32, 48, 64, 96, 128)
2. Compute perplexity on the **real** next sentence
3. Compute perplexity on **impostor** sentences from other documents in the same genre
4. **Contrast score** = log(ppl_impostor / ppl_real)

Higher contrast = the model better knows this sentence belongs here, given the context.

## Key Design Choices
- **Impostors from other documents in the same genre** — controls for topic/vocabulary while testing document-specific coherence
- **Sentence-shuffled control** — same sentences in random order, tests whether sequential ordering matters
- **Why sentence-level?** Token-level metrics (perplexity, top-k accuracy) are dominated by local syntax and vocabulary estimation. Sentence-level contrast captures discourse coherence — the "which city are you in" level, not "which restaurant did you visit."

## Expected Results
- Contrast should increase monotonically with context window
- Shuffled text should show less benefit from context than intact text
- The difference = sequential coherence beyond topic identification

## Setup
Install dependencies and import libraries. Runs on Google Colab with GPU.

In [ ]:
!pip install -q -U bitsandbytes>=0.46.1 accelerate

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from pathlib import Path
import json, math, time, gc, os, re, torch
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

print("Imports OK")

## Configuration
Set paths, model name, context windows, and parameters.
- `WINDOWS`: context sizes to test
- `N_IMPOSTORS`: how many impostor sentences per comparison (more = less noisy, but slower)
- Impostors come from other documents in the same genre

In [ ]:
IN_COLAB = 'COLAB_GPU' in os.environ or os.path.exists('/content')

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_DATA = Path("/content/drive/MyDrive/LRTIA/Data/raid_sampled")
    DRIVE_RESULTS = Path("/content/drive/MyDrive/LRTIA/Results/RAID_v5")
    if (DRIVE_DATA / "raid_corpus.jsonl").exists():
        DATA_DIR = DRIVE_DATA
    else:
        LOCAL_DATA = Path("/content/data/raid_sampled")
        if not (LOCAL_DATA / "raid_corpus.jsonl").exists():
            LOCAL_DATA.mkdir(parents=True, exist_ok=True)
            from google.colab import files
            uploaded = files.upload()
            for fname in uploaded:
                with open(LOCAL_DATA / fname, 'wb') as f:
                    f.write(uploaded[fname])
        DATA_DIR = LOCAL_DATA
    BASE_DIR = DRIVE_RESULTS
    BASE_DIR.mkdir(parents=True, exist_ok=True)
else:
    BASE_DIR = Path("../results/raid_v5")
    DATA_DIR = Path("../data/raid_sampled")
    BASE_DIR.mkdir(parents=True, exist_ok=True)

print(f"DATA_DIR: {DATA_DIR}")
print(f"BASE_DIR: {BASE_DIR}")

MODEL_NAME = "mistralai/Mistral-7B-v0.1"
USE_4BIT = True

WINDOWS = [4, 8, 12, 16, 24, 32, 48, 64, 96, 128]
N_IMPOSTORS = 5
N_RANDOM_CONTROLS = 30
RANDOM_SEED = 42

DOMAINS = ['abstracts', 'books', 'news', 'poetry', 'recipes', 'reddit', 'reviews', 'wiki']

print(f"Windows: {WINDOWS}")
print(f"Impostors per boundary: {N_IMPOSTORS}")

## Load Corpus
Load RAID dataset, filter to human-only for speed, and build impostor sentence pools organized by genre. Each genre has a pool of sentences from all its documents — impostors for a given document are drawn from other documents' pools.

In [ ]:
corpus_path = DATA_DIR / "raid_corpus.jsonl"
corpus = []
with open(corpus_path) as f:
    for line in f:
        corpus.append(json.loads(line))
print(f"Loaded {len(corpus)} documents")

# Human only for speed
corpus = [d for d in corpus if d['population'] == 'human']
print(f"Filtered to human only: {len(corpus)} docs")

# Build impostor sentence pools per domain (from OTHER documents)
print("Building sentence pools...")
sentence_pools = {d: {} for d in DOMAINS}  # domain -> {doc_id: [sentences]}
for doc in corpus:
    sentences = re.split(r'(?<=[.!?])\s+', doc['text'].strip())
    sentences = [s.strip() for s in sentences if len(s.strip().split()) >= 5]
    if len(sentences) >= 3:
        sentence_pools[doc['domain']][doc['doc_id']] = sentences

for d in DOMAINS:
    n_docs = len(sentence_pools[d])
    n_sents = sum(len(v) for v in sentence_pools[d].values())
    print(f"  {d:<15}: {n_docs} docs, {n_sents} sentences")
print("Done")

## Load Model
Load Mistral-7B with 4-bit quantization. This is the **measuring instrument** — it reads the text and evaluates how predictable each sentence is given the context.

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

if USE_4BIT:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, quantization_config=bnb_config, device_map="auto")
else:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, torch_dtype=torch.float16, device_map="auto")
model.eval()
print("Model loaded")

## Core Functions

**`compute_ppl_on_tokens(context_ids, target_ids)`** — Given context tokens and target tokens, compute the perplexity of the target. This is the fundamental measurement.

**`split_into_sentences(text, tokenizer)`** — Split text into sentences and map to token positions.

**`get_impostor_sentences(domain, exclude_doc_id, rng, n)`** — Sample impostor sentences from other documents in the same genre.

**`compute_contrast_curve(doc, rng)`** — The main computation. For each target sentence × each context window: compute perplexity on the real sentence, compute perplexity on N impostor sentences, calculate the log ratio (contrast score).

In [ ]:
@torch.no_grad()
def compute_ppl_on_tokens(context_ids, target_ids):
    """Compute perplexity of target_ids given context_ids as prefix."""
    full_ids = context_ids + target_ids
    input_ids = torch.tensor([full_ids], device=model.device)
    outputs = model(input_ids)
    logits = outputs.logits[0]

    target_start = len(context_ids)
    total_loss = 0.0
    count = 0
    for i in range(target_start, len(full_ids) - 1):
        log_probs = torch.log_softmax(logits[i], dim=-1)
        total_loss += -log_probs[full_ids[i + 1]].item()
        count += 1

    del outputs, logits
    torch.cuda.empty_cache()

    return math.exp(total_loss / count) if count > 0 else float('inf')


def split_into_sentences(text, tokenizer):
    """Split text into sentences with token positions."""
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    sentences = [s.strip() for s in sentences if len(s.strip()) > 0]

    result = []
    current_pos = 0
    for sent_text in sentences:
        sent_ids = tokenizer.encode(sent_text, add_special_tokens=False)
        if len(sent_ids) >= 3:
            result.append({
                'text': sent_text,
                'ids': sent_ids,
                'start': current_pos,
                'end': current_pos + len(sent_ids),
            })
        current_pos += len(sent_ids)

    return result


def get_impostor_sentences(domain, exclude_doc_id, rng, n=5):
    """Get impostor sentences from OTHER documents in the same genre."""
    pool = sentence_pools.get(domain, {})
    # Collect sentences from all docs except this one
    candidates = []
    for doc_id, sents in pool.items():
        if doc_id != exclude_doc_id:
            candidates.extend(sents)
    if len(candidates) < n:
        return []
    chosen = rng.choice(candidates, size=n, replace=False)
    return list(chosen)


def compute_contrast_curve(doc, rng):
    """Compute sentence contrast scores at each context window.
    Impostors are sentences from OTHER documents in the same genre."""
    text = doc['text']
    domain = doc['domain']
    doc_id = doc['doc_id']
    full_ids = tokenizer.encode(text, add_special_tokens=False)
    n_tokens = len(full_ids)

    if n_tokens < 200:
        return None

    sentences = split_into_sentences(text, tokenizer)
    if len(sentences) < 4:
        return None

    # Use sentences in the latter half as targets
    mid = len(sentences) // 2
    target_sents = sentences[mid:]

    if len(target_sents) < 2:
        return None

    results_by_W = {}

    for W in WINDOWS:
        contrast_scores = []

        for tgt in target_sents:
            sent_start = tgt['start']
            sent_end = tgt['end']

            if sent_end > n_tokens or sent_start < W:
                continue

            context_start = max(0, sent_start - W)
            context_ids = full_ids[context_start:sent_start]
            real_sent_ids = tgt['ids']

            if len(real_sent_ids) < 3 or len(context_ids) < 2:
                continue

            # Perplexity of real continuation
            ppl_real = compute_ppl_on_tokens(context_ids, real_sent_ids)
            if math.isinf(ppl_real) or ppl_real <= 0:
                continue

            # Get impostor sentences from other docs in same genre
            imp_texts = get_impostor_sentences(domain, doc_id, rng, n=N_IMPOSTORS)
            if len(imp_texts) < 2:
                continue

            ppl_impostors = []
            for imp_text in imp_texts:
                imp_ids = tokenizer.encode(imp_text, add_special_tokens=False)
                if len(imp_ids) < 3:
                    continue
                ppl_imp = compute_ppl_on_tokens(context_ids, imp_ids)
                if not math.isinf(ppl_imp) and ppl_imp > 0:
                    ppl_impostors.append(ppl_imp)

            if len(ppl_impostors) < 2:
                continue

            mean_imp_ppl = np.mean(ppl_impostors)
            contrast = math.log(mean_imp_ppl / ppl_real)
            contrast_scores.append(contrast)

        if len(contrast_scores) >= 2:
            results_by_W[W] = {
                'contrast_mean': np.mean(contrast_scores),
                'contrast_median': np.median(contrast_scores),
                'contrast_std': np.std(contrast_scores),
                'n_sentences': len(contrast_scores),
            }

    return results_by_W if len(results_by_W) >= 3 else None


def extract_row(doc, curve, n_tokens):
    row = {
        'doc_id': doc['doc_id'],
        'domain': doc['domain'],
        'model': doc['model'],
        'population': doc['population'],
        'token_count': n_tokens,
    }
    for W, metrics in sorted(curve.items()):
        row[f'contrast_W{W}'] = metrics['contrast_mean']
        row[f'contrast_med_W{W}'] = metrics['contrast_median']
        row[f'n_sent_W{W}'] = metrics['n_sentences']
    return row


print("Functions defined")

## Run Main Analysis
Compute contrast curves for all documents. This is the expensive step — each document requires (N_target_sentences × N_windows × (1 + N_impostors)) forward passes through the model.

Results are saved to CSV for resume support.

In [ ]:
results_path = BASE_DIR / "essay_results_v5.csv"

if results_path.exists():
    df = pd.read_csv(results_path)
    print(f"Loaded existing results: {len(df)} rows")
else:
    rng = np.random.RandomState(RANDOM_SEED)
    results = []
    skipped = 0
    for doc in tqdm(corpus, desc="Essays"):
        token_ids = tokenizer.encode(doc["text"], add_special_tokens=False)
        if len(token_ids) < 200:
            skipped += 1
            continue
        curve = compute_contrast_curve(doc, rng)
        if curve is None:
            skipped += 1
            continue
        results.append(extract_row(doc, curve, len(token_ids)))

    df = pd.DataFrame(results)
    df.to_csv(results_path, index=False)
    print(f"Processed {len(df)} essays ({skipped} skipped), saved to {results_path}")

print(f"Human: {len(df[df.population == 'human'])}, AI: {len(df[df.population == 'ai'])}")

## Run Controls
Sentence-shuffled versions of human documents — same sentences in random order. If context helps equally for shuffled and intact text, the signal is just topic identification. The difference between intact and shuffled = sequential coherence.

In [ ]:
# Controls: sentence-shuffled within document
controls_path = BASE_DIR / "control_results_v5.csv"

if controls_path.exists():
    df_ctrl = pd.read_csv(controls_path)
    print(f"Loaded existing controls: {len(df_ctrl)} rows")
else:
    rng_ctrl = np.random.RandomState(RANDOM_SEED)
    sample_docs = rng_ctrl.choice(corpus, size=min(N_RANDOM_CONTROLS, len(corpus)), replace=False)
    ctrl_rows = []

    rng_s = np.random.RandomState(RANDOM_SEED)
    for i, doc in enumerate(tqdm(sample_docs, desc="Shuffled (sentence-order)")):
        # Shuffle sentence order within the document
        sentences = re.split(r'(?<=[.!?])\s+', doc['text'].strip())
        sentences = [s.strip() for s in sentences if len(s.strip()) > 0]
        if len(sentences) < 6:
            continue
        rng_s.shuffle(sentences)
        shuffled_text = ' '.join(sentences)

        shuffled_doc = {
            'doc_id': f'shuffled_{i:03d}',
            'domain': doc['domain'],
            'model': 'shuffled',
            'population': 'shuffled',
            'text': shuffled_text,
        }

        curve = compute_contrast_curve(shuffled_doc, rng_ctrl)
        if curve is None:
            continue
        token_ids = tokenizer.encode(shuffled_text, add_special_tokens=False)
        ctrl_rows.append(extract_row(shuffled_doc, curve, len(token_ids)))

    df_ctrl = pd.DataFrame(ctrl_rows)
    df_ctrl.to_csv(controls_path, index=False)
    print(f"Controls: {len(df_ctrl)} rows, saved to {controls_path}")

print(f"Shuffled: {len(df_ctrl[df_ctrl.population == 'shuffled'])}")

## Visualization & Analysis
- **Panel A**: Raw contrast by condition (human vs shuffled)
- **Panel B**: Corrected contrast (human minus shuffled) — the pure coherence signal
- **Panel C**: Normalized curves by genre — is the pattern universal?
- **Panel D**: Marginal gain per token — the influence decay function

In [ ]:
human = df[df.population == 'human']
ai = df[df.population == 'ai']
shuf = df_ctrl[df_ctrl.population == 'shuffled'] if len(df_ctrl) > 0 else pd.DataFrame()

contrast_cols = [f'contrast_W{w}' for w in WINDOWS]

def get_means(sub, cols):
    return np.array([sub[c].mean() for c in cols if c in sub.columns])

fig, axes = plt.subplots(2, 2, figsize=(14, 11))

# A: Raw contrast scores
ax = axes[0, 0]
for sub, label, color, ls in [
    (human, 'Human', '#3498db', '-'),
    (ai, 'AI', '#e74c3c', '--'),
    (shuf, 'Shuffled', '#2ecc71', ':'),
]:
    if len(sub) == 0:
        continue
    vals = get_means(sub, contrast_cols)
    ax.plot(WINDOWS[:len(vals)], vals, f'o{ls}', color=color, linewidth=2, markersize=5, label=label)
ax.set_xscale('log', base=2)
ax.set_xlabel('Context Window (tokens)')
ax.set_ylabel('Contrast Score\nlog(ppl_impostor / ppl_real)')
ax.set_title('A. Sentence Contrast by Condition', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.2)

# B: Corrected (human - shuffled)
ax = axes[0, 1]
if len(shuf) > 0:
    h_vals = get_means(human, contrast_cols)
    a_vals = get_means(ai, contrast_cols)
    s_vals = get_means(shuf, contrast_cols)
    n = min(len(h_vals), len(s_vals))
    ax.plot(WINDOWS[:n], h_vals[:n] - s_vals[:n], 'o-', color='#3498db', linewidth=2, markersize=5, label='Human')
    ax.plot(WINDOWS[:n], a_vals[:n] - s_vals[:n], 'o--', color='#e74c3c', linewidth=2, markersize=5, label='AI')
    ax.set_xscale('log', base=2)
    ax.set_xlabel('Context Window (tokens)')
    ax.set_ylabel('Corrected Contrast\n(raw − shuffled)')
    ax.set_title('B. Coherence Signal (shuffled removed)', fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.2)
else:
    ax.text(0.5, 0.5, 'No shuffled controls', ha='center', va='center', transform=ax.transAxes)

# C: Normalized contrast curves by genre
ax = axes[1, 0]
colors_genre = plt.cm.Set2.colors
for i, d in enumerate(DOMAINS):
    sub = human[human.domain == d]
    vals = get_means(sub, contrast_cols)
    if len(vals) >= 3:
        total = vals[-1] - vals[0]
        if total > 0.01:
            norm = (vals - vals[0]) / total
            ax.plot(WINDOWS[:len(norm)], norm, 'o-', color=colors_genre[i], linewidth=1.5, markersize=3, label=d)

ax.axhline(0.5, color='gray', linestyle=':', alpha=0.4)
ax.plot([WINDOWS[0], WINDOWS[-1]], [0, 1], 'k--', alpha=0.2)
ax.set_xscale('log', base=2)
ax.set_ylim(-0.1, 1.1)
ax.set_xlabel('Context Window (tokens)')
ax.set_ylabel('Fraction of Total Contrast Gain')
ax.set_title('C. Contrast Curve by Genre (human)', fontweight='bold')
ax.legend(fontsize=7, ncol=2)
ax.grid(True, alpha=0.2)

# D: Marginal gain
ax = axes[1, 1]
for sub, label, color, ls in [(human, 'Human', '#3498db', '-'), (ai, 'AI', '#e74c3c', '--')]:
    vals = get_means(sub, contrast_cols)
    marg = []
    marg_x = []
    for i in range(1, len(vals)):
        dw = WINDOWS[i] - WINDOWS[i-1]
        dv = vals[i] - vals[i-1]
        marg.append(dv / dw)
        marg_x.append((WINDOWS[i] + WINDOWS[i-1]) / 2)
    ax.plot(marg_x, marg, f'o{ls}', color=color, linewidth=1.5, markersize=4, label=label)
ax.axhline(0, color='gray', linestyle=':', alpha=0.4)
ax.set_xscale('log', base=2)
ax.set_xlabel('Context Distance (tokens)')
ax.set_ylabel('Marginal Contrast Gain per Token')
ax.set_title('D. Influence Decay (sentence-level)', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.2)

plt.suptitle('Sentence Contrast Memory Curves: Does Distant Context Help Identify the Right Continuation?',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(BASE_DIR / 'fig1_contrast.png', dpi=150, bbox_inches='tight')
plt.show()

# Print raw values
print('\nContrast scores (mean):')
print(f'{"W":<6} {"Human":>10} {"AI":>10} {"Shuffled":>10}')
print('-'*40)
for i, w in enumerate(WINDOWS):
    col = f'contrast_W{w}'
    h = human[col].mean() if col in human.columns else np.nan
    a = ai[col].mean() if col in ai.columns else np.nan
    s = shuf[col].mean() if col in shuf.columns and len(shuf) > 0 else np.nan
    print(f'{w:<6} {h:>10.4f} {a:>10.4f} {s:>10.4f}')